In [1]:
import numpy as np
import jax.numpy as jnp
import jax
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

from splank import SoftPlankton, Flow, FluidPlankton

np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

## Assembly creation

### Data 

In [2]:
yaml_data = """
# Variable Prefixes (Optional)
dof_names:        # DOF prefixes/names (extends default "dof")
    - x           # Recognized as a DOF prefix/name (e.g., "x0" in expressions)
param_names:      # Parameter prefixes/names (extends default "param" and "k")
    - myradius    # Recognized as a parameter prefix/name (e.g., `myradius` in expressions)

# Default Values (Optional)
defaults:
  myradius0: 1.   # Parameter: Listed in `param_names`
  myradius1: 1.
  k: 1            # Parameter: Auto-detected (default prefix "k")

# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: myradius0                       
    mass: 0
    position: [-1, 0, 0]             
    force: [k*x, 0, 0]              

  - # Sphere 1 #################
    radius: myradius1                 
    mass: 0
    position: [1+x, 0, 0]       
    force: [-k*x, 0, 0]
"""

In [3]:
myplankton= SoftPlankton(yaml_data, verbose=False)

print(repr(myplankton))

SPHERE ASSEMBLY
  2 spheres
  1 degrees of freedom
  3 fixed parameters

Default values
  degrees of freedom dof: ['x'] = [ 0.]
  fixed parameters param: ['k', 'myradius0', 'myradius1'] = [ 1.  1.  1.]

SPHERE 0
  radius: 1.0
  mass: 0.0
  position: [-1.  0.  0.]
  orientation: [ 0.  0.  0.]
  force: [ 0.  0.  0.]
  torque: [ 0.  0.  0.]

SPHERE 1
  radius: 1.0
  mass: 0.0
  position: [ 1.  0.  0.]
  orientation: [ 0.  0.  0.]
  force: [-0.  0.  0.]
  torque: [ 0.  0.  0.]



In [ ]:
#@jax.jit 
def compute_matrix(dofs):
    params=jnp.array([1, 1, 1])
    # J = myplankton.compute_Jacobian_matrix(dofs, params)
    # R = myplankton.compute_resistance_tensor(dofs, params)
    # Mred = jnp.linalg.inv(jnp.einsum("ji, jk, kl->il", J, R, J))
    
    matrices = myplankton.compute_full_mobility_problem(dofs, params)
    
    return matrices.Mm

In [5]:
mydof = jnp.array([1.])
compute_matrix(mydof).block_until_ready()

%timeit compute_matrix(mydof).block_until_ready()

7.73 µs ± 208 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [6]:
compute_matrix_jit = jax.jit(compute_matrix)

# Pre-compile the function before timing...
compute_matrix_jit(mydof).block_until_ready()

%timeit compute_matrix_jit(mydof).block_until_ready()

7.92 µs ± 71 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
